In [ ]:
!pip install deep-translator nlpaug transformers sentencepiece


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
pip install scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 347.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 201.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 187.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import time  # MISSING IMPORT - you need this!
from deep_translator import GoogleTranslator
from transformers import pipeline
from tqdm import tqdm

class TamilDataEngine:
    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)
        print(f"✅ Loaded {len(self.df)} samples.")
        print("Original Distribution:\n", self.df['labels'].value_counts())

        # Initialize Translators
        self.to_english = GoogleTranslator(source='ta', target='en')
        self.to_tamil = GoogleTranslator(source='en', target='ta')

        # Use IndicBART for paraphrasing
        self.paraphraser = pipeline("text2text-generation",
                                    model="ai4bharat/IndicBART",
                                    device=0)

    def back_translate(self, text):
        """Tamil -> English -> Tamil back-translation"""
        try:
            en_text = self.to_english.translate(text)
            time.sleep(0.5)  # Rate limiting
            ta_text = self.to_tamil.translate(en_text)
            return ta_text
        except:
            return text

    def paraphrase_indic(self, text):
        """Use IndicBART for paraphrasing"""
        try:
            result = self.paraphraser(f"paraphrase: {text}", max_length=128)
            return result[0]['generated_text']
        except:
            return text

    def run_augmentation(self):
        augmented_data = []

        for label in self.df['labels'].unique():
            class_df = self.df[self.df['labels'] == label]
            current_count = len(class_df)

            if label not in ['Substantiated', 'Negative', 'Positive', 'Neutral', 'Sarcastic']:
                augmented_data.append(class_df)
                continue

            print(f"\n🚀 Augmenting {label}: {current_count} → 1000")

            needed = 1000 - current_count
            if needed <= 0:
                augmented_data.append(class_df)
                continue

            generated = []
            src_texts = class_df['content'].tolist()

            with tqdm(total=needed) as pbar:
                while len(generated) < needed:
                    for text in src_texts:
                        if len(generated) >= needed:
                            break

                        # Randomly choose augmentation method
                        if np.random.rand() > 0.5:
                            new_text = self.back_translate(text)
                        else:
                            new_text = self.paraphrase_indic(text)

                        if new_text and new_text != text:
                            generated.append({'content': new_text, 'labels': label})
                            pbar.update(1)

            augmented_data.append(class_df)
            augmented_data.append(pd.DataFrame(generated))

        final_df = pd.concat(augmented_data).sample(frac=1).reset_index(drop=True)
        final_df.to_csv('PS_train_augmented.csv', index=False)
        print(f"\n✨ Final Size: {len(final_df)}")
        print(final_df['labels'].value_counts())

# ===== EXECUTION SECTION (ADD THIS) =====
if __name__ == "__main__":
    # Create engine and run augmentation
    engine = TamilDataEngine('task 3 train.csv')
    engine.run_augmentation()

✅ Loaded 4352 samples.
Original Distribution:
 labels
Opinionated          1361
Sarcastic             790
Neutral               637
Positive              575
Substantiated         412
Negative              406
None of the above     171
Name: count, dtype: int64


Device set to use cuda:0



🚀 Augmenting Neutral: 637 → 1000


  6%|▌         | 22/363 [00:32<06:04,  1.07s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 57%|█████▋    | 206/363 [04:54<05:40,  2.17s/it]/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1035,0,0], thread: [224,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1035,0,0], thread: [225,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [1035,0,


🚀 Augmenting Substantiated: 412 → 1000


...


🚀 Augmenting Sarcastic: 790 → 1000


100%|██████████| 210/210 [07:24<00:00,  2.12s/it]



🚀 Augmenting Negative: 406 → 1000


100%|██████████| 594/594 [16:05<00:00,  1.63s/it]


✨ Final Size: 6532
labels
Opinionated          1361
Negative             1000
Sarcastic            1000
Positive             1000
Neutral              1000
Substantiated        1000
None of the above     171
Name: count, dtype: int64


In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# FIX 1: Import AdamW from torch.optim (Hugging Face version is deprecated)
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# --- CONFIGURATION ---
# FIX 2: Use the stable BERT model (V3 is a Gemma-LLM and requires different code)
MODEL_NAME = "ai4bharat/indic-bert"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5
DATA_PATH = 'PS_train_augmented.csv' # Ensure this file exists!

# --- 1. CUSTOM DATASET ---
class DravidianDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# --- 2. CUSTOM MODEL ARCHITECTURE ---
class PoliticalSentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(PoliticalSentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)

        # Adjust hidden size automatically
        hidden_size = self.bert.config.hidden_size
        self.fc1 = nn.Linear(hidden_size, 768)
        self.relu = nn.ReLU()
        self.out = nn.Linear(768, n_classes)

    def forward(self, input_ids, attention_mask):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Safe pooling for IndicBERT
        if hasattr(output, 'pooler_output') and output.pooler_output is not None:
             x = output.pooler_output
        else:
             # Fallback for models without pooler (Mean Pooling)
             x = output.last_hidden_state[:, 0, :]

        x = self.drop(x)
        x = self.fc1(x)
        x = self.relu(x)
        return self.out(x)

# --- 3. EXECUTION LOGIC ---
if __name__ == "__main__":
    # Check if data exists
    if not os.path.exists(DATA_PATH):
        print(f"⚠️ Warning: {DATA_PATH} not found. Using 'PS_train - PS_train.csv' instead.")
        DATA_PATH = 'PS_train - PS_train.csv'

    # Load Data
    df = pd.read_csv(DATA_PATH)

    # Encode Labels
    le = LabelEncoder()
    df['encoded_labels'] = le.fit_transform(df['labels'])
    num_classes = len(le.classes_)
    print(f"✅ Classes found: {le.classes_}")

    # Calculate Class Weights
    class_counts = df['encoded_labels'].value_counts().sort_index().values
    total_samples = len(df)
    # FIX 3: Add small epsilon to avoid division by zero
    class_weights = total_samples / (num_classes * (class_counts + 1e-5))
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    # Split
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['encoded_labels'])

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Dataloaders
    train_dataset = DravidianDataset(train_df['content'].values, train_df['encoded_labels'].values, tokenizer, MAX_LEN)
    val_dataset = DravidianDataset(val_df['content'].values, val_df['encoded_labels'].values, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Setup Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Using device: {device}")

    # Initialize Model
    model = PoliticalSentimentClassifier(num_classes)
    model = model.to(device)

    # Optimizer & Scheduler
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE) # Corrected AdamW
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    # Loss Function
    loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))

    # Training Loop
    best_f1 = 0
    print("\nStarting Training...")

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        correct_predictions = 0

        for d in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)

            loss = loss_fn(outputs, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())
            correct_predictions += torch.sum(preds == targets)

        # Validation
        model.eval()
        val_losses = []
        all_preds = []
        all_true = []

        with torch.no_grad():
            for d in val_loader:
                input_ids = d["input_ids"].to(device)
                attention_mask = d["attention_mask"].to(device)
                targets = d["labels"].to(device)

                outputs = model(input_ids, attention_mask)
                _, preds = torch.max(outputs, dim=1)
                loss = loss_fn(outputs, targets)

                val_losses.append(loss.item())
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(targets.cpu().numpy())

        # Metrics
        train_acc = correct_predictions.double() / len(train_df)
        val_acc = np.mean(np.array(all_preds) == np.array(all_true))
        macro_f1 = f1_score(all_true, all_preds, average='macro')

        print(f" Train Loss: {np.mean(train_losses):.4f} | Acc: {train_acc:.4f}")
        print(f" Val Loss:   {np.mean(val_losses):.4f} | Acc: {val_acc:.4f} | Macro F1: {macro_f1:.4f}")

        if macro_f1 > best_f1:
            torch.save(model.state_dict(), 'best_model.bin')
            best_f1 = macro_f1
            print(" 🏆 New Best Model Saved!")

    print("\nFinal Classification Report:")
    print(classification_report(all_true, all_preds, target_names=le.classes_))

✅ Classes found: ['Negative' 'Neutral' 'None of the above' 'Opinionated' 'Positive'
 'Sarcastic' 'Substantiated']


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

🚀 Using device: cuda


2026-01-20 19:27:06.019473: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-20 19:27:06.053514: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-20 19:27:13.676223: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


...

 Train Loss: 1.5618 | Acc: 0.3071
 Val Loss:   1.5854 | Acc: 0.2997 | Macro F1: 0.3233
 🏆 New Best Model Saved!


Epoch 4/5: 100%|██████████| 368/368 [00:30<00:00, 12.18it/s]


 Train Loss: 1.4881 | Acc: 0.3426
 Val Loss:   1.5644 | Acc: 0.3471 | Macro F1: 0.3704
 🏆 New Best Model Saved!


Epoch 5/5: 100%|██████████| 368/368 [00:30<00:00, 11.88it/s]


 Train Loss: 1.4112 | Acc: 0.3831
 Val Loss:   1.5590 | Acc: 0.3379 | Macro F1: 0.3675

Final Classification Report:
                   precision    recall  f1-score   support

         Negative       0.24      0.14      0.18       100
          Neutral       0.33      0.26      0.29       100
None of the above       0.79      0.88      0.83        17
      Opinionated       0.35      0.61      0.44       137
         Positive       0.30      0.19      0.23       100
        Sarcastic       0.38      0.15      0.22       100
    Substantiated       0.31      0.49      0.38       100

         accuracy                           0.34       654
        macro avg       0.39      0.39      0.37       654
     weighted avg       0.33      0.34      0.31       654



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# --- UPGRADE 1: STRONGER MODEL ---
MODEL_NAME = "xlm-roberta-base" # Much more powerful than IndicBERT
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 6  # Added 1 epoch for the larger model
LEARNING_RATE = 1e-5 # Slower learning rate for stability
DATA_PATH = '/workspace/PS_train_augmented.csv'

# --- UPGRADE 2: FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Flatten label for gathering
        targets = targets.view(-1, 1)

        # Calculate log probabilities
        logpt = F.log_softmax(inputs, dim=1)

        # Get log probability of the true class
        logpt = logpt.gather(1, targets)
        logpt = logpt.view(-1)
        pt = logpt.exp()

        # Weighting
        if self.alpha is not None:
            at = self.alpha.gather(0, targets.view(-1))
            logpt = logpt * at

        # Focal term: (1 - pt)^gamma
        loss = -1 * (1 - pt)**self.gamma * logpt

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# --- DATASET (Standard) ---
class DravidianDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        # Calling the tokenizer directly is more robust than .encode_plus
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# --- MODEL ARCHITECTURE (XLM-R Version) ---
class PoliticalSentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(PoliticalSentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # XLM-R uses the CLS token (index 0) similar to BERT
        pooled_output = output.last_hidden_state[:, 0, :]
        x = self.drop(pooled_output)
        return self.out(x)

# --- EXECUTION ---
if __name__ == "__main__":
    if not os.path.exists(DATA_PATH):
        print(f"⚠️ Warning: {DATA_PATH} not found. Using original data.")
        DATA_PATH = '/workspace/task 3 train.csv'

    df = pd.read_csv(DATA_PATH)

    # Encode
    le = LabelEncoder()
    df['encoded_labels'] = le.fit_transform(df['labels'])
    num_classes = len(le.classes_)

    # Calculate Alpha for Focal Loss (Inverse Class Frequency)
    class_counts = df['encoded_labels'].value_counts().sort_index().values
    weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    weights = weights / weights.sum() # Normalize

    # Split
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['encoded_labels'])

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_dataset = DravidianDataset(train_df['content'].values, train_df['encoded_labels'].values, tokenizer, MAX_LEN)
    val_dataset = DravidianDataset(val_df['content'].values, val_df['encoded_labels'].values, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Upgraded to {MODEL_NAME} on {device}")

    model = PoliticalSentimentClassifier(num_classes).to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    # INITIALIZE FOCAL LOSS
    criterion = FocalLoss(alpha=weights.to(device), gamma=2.0)

    best_f1 = 0
    print("\nStarting Advanced Training...")

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        all_preds = []
        all_true = []

        for d in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)

            loss = criterion(outputs, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_preds = []
        val_true = []

        with torch.no_grad():
            for d in val_loader:
                input_ids = d["input_ids"].to(device)
                attention_mask = d["attention_mask"].to(device)
                targets = d["labels"].to(device)

                outputs = model(input_ids, attention_mask)
                _, preds = torch.max(outputs, dim=1)

                val_preds.extend(preds.cpu().numpy())
                val_true.extend(targets.cpu().numpy())

        macro_f1 = f1_score(val_true, val_preds, average='macro')
        print(f" Epoch {epoch+1} | Train Loss: {np.mean(train_losses):.4f} | Val Macro F1: {macro_f1:.4f}")

        if macro_f1 > best_f1:
            torch.save(model.state_dict(), 'xlmr_focal_best.bin')
            best_f1 = macro_f1
            print(" 🏆 New Best Model Saved!")

    print("\nFinal Report:")
    print(classification_report(val_true, val_preds, target_names=le.classes_))

🚀 Upgraded to xlm-roberta-base on cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Starting Advanced Training...


Epoch 1: 100%|██████████| 368/368 [00:42<00:00,  8.66it/s]


...

 Epoch 4 | Train Loss: 0.0937 | Val Macro F1: 0.4248
 🏆 New Best Model Saved!


Epoch 5: 100%|██████████| 368/368 [00:42<00:00,  8.68it/s]


 Epoch 5 | Train Loss: 0.0881 | Val Macro F1: 0.4222


Epoch 6: 100%|██████████| 368/368 [00:42<00:00,  8.71it/s]


 Epoch 6 | Train Loss: 0.0846 | Val Macro F1: 0.4268
 🏆 New Best Model Saved!

Final Report:
                   precision    recall  f1-score   support

         Negative       0.27      0.29      0.28       100
          Neutral       0.52      0.12      0.20       100
None of the above       0.94      0.88      0.91        17
      Opinionated       0.34      0.45      0.39       137
         Positive       0.31      0.48      0.38       100
        Sarcastic       0.54      0.48      0.51       100
    Substantiated       0.37      0.30      0.33       100

         accuracy                           0.37       654
        macro avg       0.47      0.43      0.43       654
     weighted avg       0.40      0.37      0.36       654



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import gc

# --- CONFIGURATION ---
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-5
FOLDS = 5  # Standard for competitions
DATA_PATH = '/workspace/PS_train_augmented.csv'

# --- FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        targets = targets.view(-1, 1)
        logpt = F.log_softmax(inputs, dim=1)
        logpt = logpt.gather(1, targets)
        logpt = logpt.view(-1)
        pt = logpt.exp()
        if self.alpha is not None:
            at = self.alpha.gather(0, targets.view(-1))
            logpt = logpt * at
        loss = -1 * (1 - pt)**self.gamma * logpt
        if self.reduction == 'mean':
            return loss.mean()
        else:
            return loss

# --- DATASET ---
class DravidianDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        # Calling the tokenizer directly is more robust than .encode_plus
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }


# --- MODEL ---
class PoliticalSentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(PoliticalSentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.4) # Increased Dropout for regularization
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = output.last_hidden_state[:, 0, :]
        x = self.drop(pooled_output)
        return self.out(x)

# --- EXECUTION ---
if __name__ == "__main__":
    if not os.path.exists(DATA_PATH):
        DATA_PATH = '/workspace/task 3 train.csv'

    df = pd.read_csv(DATA_PATH)

    # Encode
    le = LabelEncoder()
    df['encoded_labels'] = le.fit_transform(df['labels'])
    num_classes = len(le.classes_)

    # Calculate Alpha for Focal Loss
    class_counts = df['encoded_labels'].value_counts().sort_index().values
    weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    weights = weights / weights.sum()

    # Initialize Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_preds = []
    oof_targets = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Starting {FOLDS}-Fold Cross Validation on {device}...")

    # --- CV LOOP ---
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['encoded_labels'])):
        print(f"\n=== FOLD {fold+1}/{FOLDS} ===")

        # Prepare Data for this Fold
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]

        train_dataset = DravidianDataset(train_df['content'].values, train_df['encoded_labels'].values, tokenizer, MAX_LEN)
        val_dataset = DravidianDataset(val_df['content'].values, val_df['encoded_labels'].values, tokenizer, MAX_LEN)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

        # Init Model
        model = PoliticalSentimentClassifier(num_classes).to(device)
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
        total_steps = len(train_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        criterion = FocalLoss(alpha=weights.to(device), gamma=2.0)

        best_fold_f1 = 0

        for epoch in range(EPOCHS):
            model.train()
            for d in tqdm(train_loader, desc=f"Fold {fold+1} Epoch {epoch+1}", leave=False):
                input_ids = d["input_ids"].to(device)
                attention_mask = d["attention_mask"].to(device)
                targets = d["labels"].to(device)

                optimizer.zero_grad()
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, targets)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

            # Validation
            model.eval()
            val_preds = []
            val_true = []
            with torch.no_grad():
                for d in val_loader:
                    input_ids = d["input_ids"].to(device)
                    attention_mask = d["attention_mask"].to(device)
                    targets = d["labels"].to(device)
                    outputs = model(input_ids, attention_mask)
                    _, preds = torch.max(outputs, dim=1)
                    val_preds.extend(preds.cpu().numpy())
                    val_true.extend(targets.cpu().numpy())

            fold_f1 = f1_score(val_true, val_preds, average='macro')

            if fold_f1 > best_fold_f1:
                best_fold_f1 = fold_f1
                torch.save(model.state_dict(), f'xlmr_fold_{fold}.bin')

        print(f"✅ Best F1 for Fold {fold+1}: {best_fold_f1:.4f}")

        # Clean up to save memory
        del model, optimizer, scheduler
        torch.cuda.empty_cache()
        gc.collect()

    print("\n🏆 Training Complete! You now have 5 models (xlmr_fold_0.bin to xlmr_fold_4.bin).")

🚀 Starting 5-Fold Cross Validation on cuda...

=== FOLD 1/5 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Best F1 for Fold 1: 0.4158

=== FOLD 2/5 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Best F1 for Fold 4: 0.4083

=== FOLD 5/5 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Best F1 for Fold 5: 0.4069

🏆 Training Complete! You now have 5 models (xlmr_fold_0.bin to xlmr_fold_4.bin).


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import torch.nn.functional as F  # Added for softmax
from tqdm import tqdm
import os

# --- CONFIGURATION ---
MODEL_NAME = "xlm-roberta-base"
TEST_FILE = "/workspace/task_3_test.csv"         # Ensure this matches your file name
OUTPUT_FILE = "submission_ensemble.csv"
MAX_LEN = 128
BATCH_SIZE = 32
FOLDS = 5

# LABELS (Must match training order)
LABELS = [
    'Negative',
    'Neutral',
    'None of the above',
    'Opinionated',
    'Positive',
    'Sarcastic',
    'Substantiated'
]

# --- MODEL ARCHITECTURE ---
class PoliticalSentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(PoliticalSentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(p=0.4)
        self.out = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = output.last_hidden_state[:, 0, :]
        x = self.drop(pooled_output)
        return self.out(x)

# --- INFERENCE ENGINE ---
def run_ensemble_inference():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Starting Ensemble Inference on {device}")

    # 1. Load Test Data
    try:
        df = pd.read_csv(TEST_FILE)
    except:
        df = pd.read_csv(TEST_FILE, sep='\t')

    print(f"📄 Found {len(df)} test samples.")

    # Handle Column Names
    content_col = 'content'
    if content_col not in df.columns:
        if 'text' in df.columns: content_col = 'text'
        elif 'sentence' in df.columns: content_col = 'sentence'

    # 2. Tokenize (FIXED & MODERNIZED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    input_ids_list = []
    attention_masks_list = []

    print("⚙️ Tokenizing...")
    for text in tqdm(df[content_col]):
        # Call the tokenizer directly (the modern __call__ method)
        encoded = tokenizer(
            str(text),
            add_special_tokens=True,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        # We use .squeeze(0) to turn [1, 128] into [128]
        input_ids_list.append(encoded['input_ids'].squeeze(0))
        attention_masks_list.append(encoded['attention_mask'].squeeze(0))

    input_ids = torch.stack(input_ids_list)
    attention_masks = torch.stack(attention_masks_list)

    dataset = torch.utils.data.TensorDataset(input_ids, attention_masks)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 3. Ensemble Loop
    final_probs = np.zeros((len(df), len(LABELS)))

    print(f"\n🔮 Aggregating predictions from {FOLDS} models...")

    models_found = 0
    for fold in range(FOLDS):
        model_path = f"xlmr_fold_{fold}.bin"
        if not os.path.exists(model_path):
            print(f"⚠️ Warning: {model_path} missing. Skipping.")
            continue

        models_found += 1
        print(f"   -> Loading Fold {fold}...")
        model = PoliticalSentimentClassifier(len(LABELS))
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval()

        fold_probs = []

        with torch.no_grad():
            for batch in tqdm(dataloader, leave=False):
                b_ids = batch[0].to(device)
                b_mask = batch[1].to(device)

                outputs = model(b_ids, b_mask)
                probs = F.softmax(outputs, dim=1) # Softmax to get probabilities
                fold_probs.extend(probs.cpu().numpy())

        final_probs += np.array(fold_probs)

        del model
        torch.cuda.empty_cache()

    if models_found == 0:
        print("❌ Error: No trained model files (xlmr_fold_X.bin) found!")
        return

    # 4. Final Decision
    avg_probs = final_probs / models_found
    predictions = np.argmax(avg_probs, axis=1)

    final_labels = [LABELS[p] for p in predictions]

    # 5. Save Output
    submission = pd.DataFrame()

    # Try to keep ID if available
    if 'pid' in df.columns: submission['pid'] = df['pid']
    elif 'id' in df.columns: submission['id'] = df['id']
    else: submission['id'] = df.index

    submission['label'] = final_labels

    submission.to_csv(OUTPUT_FILE, sep='\t', index=False)
    print(f"\n✅ Ensemble Submission saved to {OUTPUT_FILE}")
    print(submission.head())

if __name__ == "__main__":
    run_ensemble_inference()

🚀 Starting Ensemble Inference on cuda
📄 Found 544 test samples.
⚙️ Tokenizing...


100%|██████████| 544/544 [00:00<00:00, 2891.83it/s]


🔮 Aggregating predictions from 5 models...
   -> Loading Fold 0...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_753/123890514.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be execu

...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   -> Loading Fold 4...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Ensemble Submission saved to submission_ensemble.csv
   id      label
0   0   Positive
1   1  Sarcastic
2   2   Positive
3   3   Positive
4   4   Positive
